# 03 - 集成学习方法

## 学习目标
- 理解 Bagging、Boosting、Stacking 的原理
- 掌握 RandomForest、GradientBoosting、XGBoost 的使用

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier,
    GradientBoostingClassifier, AdaBoostClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

## 1. Bagging

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

# 单棵决策树
dt = DecisionTreeClassifier(random_state=42)
dt_scores = cross_val_score(dt, X, y, cv=5)

# Bagging
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
bag_scores = cross_val_score(bagging, X, y, cv=5)

print(f'单棵树: {dt_scores.mean():.4f} ± {dt_scores.std():.4f}')
print(f'Bagging: {bag_scores.mean():.4f} ± {bag_scores.std():.4f}')

## 2. Boosting

In [ ]:
# AdaBoost
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
ada_scores = cross_val_score(ada, X, y, cv=5)

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_scores = cross_val_score(gb, X, y, cv=5)

print(f'AdaBoost: {ada_scores.mean():.4f} ± {ada_scores.std():.4f}')
print(f'GradientBoosting: {gb_scores.mean():.4f} ± {gb_scores.std():.4f}')

## 3. Stacking

In [ ]:
# 定义基学习器
estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('svm', SVC(probability=True, random_state=42))
]

# Stacking
stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5
)
stack_scores = cross_val_score(stacking, X, y, cv=5)

print(f'Stacking: {stack_scores.mean():.4f} ± {stack_scores.std():.4f}')

## 4. 模型比较

In [ ]:
models = {
    'Decision Tree': dt,
    'Bagging': bagging,
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost': ada,
    'Gradient Boosting': gb,
    'Stacking': stacking
}

results = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5)
    results.append({
        'Model': name,
        'Mean': scores.mean(),
        'Std': scores.std()
    })

df = pd.DataFrame(results).sort_values('Mean', ascending=False)
print(df.round(4).to_string(index=False))

## 小结
- Bagging 降低方差（RandomForest）
- Boosting 降低偏差（GradientBoosting）
- Stacking 组合多个模型